In [1]:
!pip install onnxruntime onnx onnxscript skl2onnx onnxmltools torch numpy pandas scikit-learn requests tqdm pyarrow xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.7 MB/s eta 0:00:00


In [1]:
from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score
import xgboost as xgb
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

CONFIG = {
    "HELIUS_API_KEY": os.environ.get("HELIUS_API_KEY", ""),
    "HELIUS_BASE_URL": "https://api.helius.xyz",
    "GRADUATION_MANIFEST_PATH": "/content/drive/MyDrive/meme_bot_training/graduated_tokens.parquet",
    "RAW_DATA_PATH": "/content/drive/MyDrive/meme_bot_training/raw_data.parquet",
    "FEATURES_PATH": "/content/drive/MyDrive/meme_bot_training/features.parquet",
    "COLLECTION_CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/collection_checkpoint.json",
    "TRAIN_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/train_dataset.pt",
    "VAL_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/val_dataset.pt",
    "TEST_CACHE_PATH": "/content/drive/MyDrive/meme_bot_training/test_dataset.pt",
    "BEST_CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model_best.pt",
    "ONNX_OUTPUT_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model.onnx",
    "DEPLOYER_LOOKUP_PATH": "/content/drive/MyDrive/meme_bot_training/deployer_lookup.json",
    "MODEL_META_PATH": "/content/drive/MyDrive/meme_bot_training/rug_model_meta.json",
    "SEQUENCE_LENGTH": 16,
    "SEQUENCE_FEATURES": ["holders", "liquidity", "volume", "ratio", "velocity", "tx_count"],
    "TABULAR_FEATURES": [
        "lpBurnGap",
        "topHolderPct",
        "devHoldPct",
        "mutableMetadata",
        "mintAuthority",
        "freezeAuthority"
    ],
    "TRAIN_START": "2023-01-01T00:00:00Z",
    "TRAIN_END": "2023-09-01T00:00:00Z",
    "VAL_START": "2023-09-01T00:00:00Z",
    "VAL_END": "2024-01-01T00:00:00Z",
    "TEST_START": "2024-01-01T00:00:00Z",
    "TEST_END": pd.Timestamp.utcnow().isoformat(),
    "LABEL_WINDOW_HOURS": 72,
    "PRICE_DROP_RUG_THRESHOLD": 0.80,
    "LIQUIDITY_DROP_RUG_THRESHOLD": 0.70,
    "LOCK_BURN_THRESHOLD": 0.90,
    "LOCK_LOCKED_THRESHOLD": 90.0,
    "ENTRY_DELAY_SECONDS": 120,
    "TIME_BUCKET_MINUTES": 5,
    "MAX_SWAP_PAGES": 200,
    "SWAP_PAGE_LIMIT": 100,
    "CHECKPOINT_EVERY_TOKENS": 100,
    "HTTP_TIMEOUT_SECONDS": 30,
    "MAX_RETRIES": 6,
    "INITIAL_BACKOFF_SECONDS": 1.5,
    "BATCH_SIZE": 256,
    "LEARNING_RATE": 1e-3,
    "WEIGHT_DECAY": 0.01,
    "EPOCHS": 40,
    "PATIENCE": 5,
    "SEED": 20260622,
    "DROPOUT_1": 0.3,
    "DROPOUT_2": 0.2,
    "AUC_QUALITY_GATE": 0.40,
    "MAX_SLIPPAGE_PCT": 1.0,
    "UNKNOWN_DEPLOYER_ID": 0,
    "EXPORT_OPSET": 15,

    "XGB_N_ESTIMATORS": 300,
    "XGB_MAX_DEPTH": 5,
    "XGB_LEARNING_RATE": 0.05,
    "XGB_SUBSAMPLE": 0.8,
    "XGB_COLSAMPLE": 0.8,
    "XGB_ENSEMBLE_WEIGHT": 0.4,
    "TORCH_ENSEMBLE_WEIGHT": 0.6,
    "XGB_MODEL_OUT": "/content/drive/MyDrive/meme_bot_training/xgb_model.onnx",

    "GITHUB_REPO": "https://github.com/Salazar254/meme_coin",
    "GITHUB_TOKEN": "",
    "AUTO_PUSH": True,

    "DRIVE_BASE": "/content/drive/MyDrive/meme_bot_training",
    "CHECKPOINT_PATH": "/content/drive/MyDrive/meme_bot_training/collection_checkpoint.json",
    "RUG_MODEL_OUT": "/content/drive/MyDrive/meme_bot_training/rug_model.onnx",
    "DEPLOYER_OUT": "/content/drive/MyDrive/meme_bot_training/deployer_lookup.json",
    "META_OUT": "/content/drive/MyDrive/meme_bot_training/rug_model_meta.json",
    "FEATURE_IMP_OUT": "/content/drive/MyDrive/meme_bot_training/feature_importance.json",
}

from google.colab import drive
drive.mount('/content/drive')

try:
    from google.colab import userdata
    CONFIG['HELIUS_API_KEY'] = userdata.get('HELIUS_API_KEY')
    CONFIG['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
except:
    print("No GITHUB_TOKEN secret found - auto-push disabled")

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# === Rate limiting & API caching ===
CACHE_PATH = CONFIG['DRIVE_BASE'] + '/api_cache.json'

def load_cache() -> dict:
    try:
        if os.path.exists(CACHE_PATH):
            with open(CACHE_PATH, 'r') as f:
                return json.load(f)
    except Exception:
        pass
    return {}

def save_cache(cache: dict) -> None:
    try:
        os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
        with open(CACHE_PATH, 'w') as f:
            json.dump(cache, f)
    except Exception:
        pass

API_CACHE = load_cache()

def api_with_retry(fn, max_retries=5):
    for attempt in range(max_retries):
        try:
            result = fn()
            if result is not None:
                return result
        except Exception:
            pass
        wait = (2 ** attempt) + random.uniform(0, 1)
        print(f"Rate limited — waiting {wait:.1f}s")
        time.sleep(wait)
    return None

# Rate limit delays between calls
RUGCHECK_DELAY = 0.6   # 100 req/min
BIRDEYE_DELAY = 1.0    # 60 req/min
HELIUS_REST_DELAY = 0.5
BATCH_SIZE_TOKENS = 50
BATCH_PAUSE = 30.0

def cached_api_call(cache_key: str, fetch_fn, delay: float = 0.0):
    """Check cache first, fetch if missing, respect rate limits."""
    if cache_key in API_CACHE:
        return API_CACHE[cache_key]
    if delay > 0:
        time.sleep(delay)
    result = api_with_retry(fetch_fn)
    if result is not None:
        API_CACHE[cache_key] = result
        save_cache(API_CACHE)
    return result

Mounted at /content/drive


In [2]:
RAW_DATA_PATH = Path(CONFIG["RAW_DATA_PATH"])
COLLECTION_CHECKPOINT_PATH = Path(CONFIG["COLLECTION_CHECKPOINT_PATH"])

def clamp(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return max(low, min(high, value))

def safe_float(value, fallback: float = 0.0) -> float:
    try:
        if value is None or value == "": return fallback
        return float(value) if math.isfinite(float(value)) else fallback
    except: return fallback

HELIUS_KEY = CONFIG["HELIUS_API_KEY"]

def fetch_dexscreener_raydium():
    """Primary: GET DexScreener Raydium pairs, no q param, filter by pairCreatedAt >= 2025-01-01"""
    url = "https://api.dexscreener.com/latest/dex/pairs/solana/raydium"
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            pairs = data.get("pairs", []) if isinstance(data, dict) else []
            # Filter by pairCreatedAt >= 2025-01-01
            cutoff_ts = datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp() * 1000
            valid = []
            for p in pairs:
                created = p.get("pairCreatedAt", 0)
                if isinstance(created, (int, float)) and created >= cutoff_ts:
                    mint = p.get("baseToken", {}).get("address", "")
                    if mint:
                        valid.append({
                            "mint": mint,
                            "deployer": "",
                            "liquidity_usd": safe_float(p.get("liquidity", {}).get("usd", 0), 0),
                            "fdv": safe_float(p.get("fdv", 0), 0),
                        })
            return valid
    except Exception as e:
        print(f"DexScreener Raydium error: {e}")
    return []

def fetch_dexscreener_token_profiles():
    """Secondary: GET DexScreener token profiles, no params."""
    url = "https://api.dexscreener.com/token-profiles/latest/v1"
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            profiles = data if isinstance(data, list) else []
            valid = []
            for p in profiles:
                mint = p.get("tokenAddress", "")
                if mint:
                    valid.append({
                        "mint": mint,
                        "deployer": "",
                        "liquidity_usd": 0,
                        "fdv": 0,
                    })
            return valid
    except Exception as e:
        print(f"DexScreener token profiles error: {e}")
    return []

def fetch_helius_pump_migrations():
    """Third: Helius pump.fun migration program transactions."""
    if not HELIUS_KEY:
        print("No HELIUS_API_KEY — skipping pump.fun migrations")
        return []
    addr = "39azUYFWPz3VHgKCf3VChUwbpURdCHRxjWVowf5jUJjg"
    url = f"https://api.helius.xyz/v0/addresses/{addr}/transactions"
    params = {"api-key": HELIUS_KEY, "limit": 100}
    try:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            txs = resp.json()
            mints = set()
            for tx in txs if isinstance(txs, list) else []:
                for transfer in tx.get("tokenTransfers", []):
                    mint = transfer.get("mint", "")
                    if mint:
                        mints.add(mint)
            return [{"mint": m, "deployer": "", "liquidity_usd": 0, "fdv": 0} for m in mints]
    except Exception as e:
        print(f"Helius pump.fun migrations error: {e}")
    return []

def collect_mints_from_apis(target_tokens: int = 200):
    """Collect mints from all sources, deduplicate."""
    all_mints = {}

    # Primary: DexScreener Raydium
    print("Fetching DexScreener Raydium pairs...")
    raydium = cached_api_call("dexscreener_raydium", fetch_dexscreener_raydium, delay=0.0)
    print(f"  Got {len(raydium) if raydium else 0} pairs")
    if raydium:
        for item in raydium:
            all_mints[item["mint"]] = item

    # Secondary: DexScreener token profiles
    print("Fetching DexScreener token profiles...")
    profiles = cached_api_call("dexscreener_profiles", fetch_dexscreener_token_profiles, delay=0.0)
    print(f"  Got {len(profiles) if profiles else 0} profiles")
    if profiles:
        for item in profiles:
            if item["mint"] not in all_mints:
                all_mints[item["mint"]] = item

    # Third: Helius pump.fun migrations
    print("Fetching Helius pump.fun migration transactions...")
    pump_mints = cached_api_call("helius_pump_migrations", fetch_helius_pump_migrations, delay=HELIUS_REST_DELAY)
    print(f"  Got {len(pump_mints) if pump_mints else 0} migrations")
    if pump_mints:
        for item in pump_mints:
            if item["mint"] not in all_mints:
                all_mints[item["mint"]] = item

    print(f"Total unique mints collected: {len(all_mints)}")
    return list(all_mints.values())

def enrich_with_birdeye_price(tokens: list):
    """Fetch Birdeye price history, use FIX 3 fallbacks on empty."""
    results = []
    for i, token in enumerate(tqdm(tokens, desc="Birdeye")):
        mint = token["mint"]
        cache_key = f"birdeye_{mint}"

        def fetch_birdeye():
            url = f"https://public-api.birdeye.so/public/price/history?address={mint}&type=15m"
            headers = {"X-API-KEY": CONFIG.get("BIRDEYE_API_KEY", "")}
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            return None

        price_data = cached_api_call(cache_key, fetch_birdeye, delay=BIRDEYE_DELAY)

        if price_data and price_data.get("data", {}).get("items"):
            items = price_data["data"]["items"]
            prices = [safe_float(item.get("value", 0)) for item in items]
            token["birdeye_prices"] = prices
            token["inferred_label"] = False
            # Compute labels from real price data
            if len(prices) > 1:
                max_price = max(prices)
                min_price = min(prices)
                if max_price > 0:
                    drawdown = (max_price - min_price) / max_price
                    token["max_drawdown_pct"] = clamp(drawdown * 100, 0, 100)
                    token["rug_label"] = 1.0 if drawdown > CONFIG["PRICE_DROP_RUG_THRESHOLD"] else 0.0
                    token["time_to_rug_hours"] = 24.0 if token["rug_label"] == 1.0 else 72.0
                    # Pump 2x check
                    first_price = prices[0]
                    token["pump_2x_label"] = 1.0 if first_price > 0 and max_price / first_price >= 2.0 else 0.0
        else:
            # FIX 3 fallback: no Birdeye price data, no rugcheck risk
            token["inferred_label"] = True
            token["rug_label"] = 0.0
            token["time_to_rug_hours"] = 72.0
            token["max_drawdown_pct"] = 20.0
            token["pump_2x_label"] = 0.0

        results.append(token)

        if (i + 1) % BATCH_SIZE_TOKENS == 0:
            print(f"  Batch pause {BATCH_PAUSE}s after {i+1} tokens...")
            time.sleep(BATCH_PAUSE)

    return results

def enrich_with_helius_transactions(tokens: list):
    """Fetch Helius transaction history for each token, use fallbacks on empty."""
    results = []
    for i, token in enumerate(tqdm(tokens, desc="Helius txs")):
        mint = token["mint"]
        cache_key = f"helius_txs_{mint}"

        def fetch_helius():
            if not HELIUS_KEY:
                return None
            url = f"https://api.helius.xyz/v0/addresses/{mint}/transactions"
            params = {"api-key": HELIUS_KEY, "limit": 50}
            resp = requests.get(url, params=params, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            return None

        txs = cached_api_call(cache_key, fetch_helius, delay=HELIUS_REST_DELAY)

        if txs and isinstance(txs, list) and len(txs) > 0:
            token["helius_transactions"] = txs
            token["has_sequence"] = True
        else:
            # FIX 3 fallback: use zero-padded sequence
            token["helius_transactions"] = []
            token["has_sequence"] = False
            token["sequence_fallback"] = True

        results.append(token)

        if (i + 1) % BATCH_SIZE_TOKENS == 0:
            print(f"  Batch pause {BATCH_PAUSE}s after {i+1} tokens...")
            time.sleep(BATCH_PAUSE)

    return results

def build_raw_df_from_tokens(tokens: list) -> pd.DataFrame:
    """Build raw_df from enriched token data."""
    rows = []
    for token in tokens:
        mint = token["mint"]
        deployer = token.get("deployer", "unknown")
        grad_ts = pd.Timestamp.utcnow() - pd.Timedelta(days=random.randint(0, 30))

        # Create 20 synthetic swap events per token for sequence building
        for i in range(20):
            rows.append({
                "mint": mint,
                "deployer": deployer,
                "graduation_timestamp": grad_ts,
                "timestamp": grad_ts - pd.Timedelta(minutes=(20 - i) * 5),
                "sol_amount": 1.0 * random.uniform(0.8, 1.2),
                "token_amount": 1000000,
                "lpBurnPct": token.get("lpBurnGap", 0.5) * 100,
                "initial_liquidity_sol": random.uniform(1, 50),
                "rugPullRisk": token.get("rugPullRisk", 0),
                "honeypotRisk": token.get("honeypotRisk", 0),
                "buyer": f"user_{random.randint(1, 100)}",
                "seller": f"user_{random.randint(1, 100)}",
                "rug_label": token.get("rug_label", 0),
                "time_to_rug_hours": token.get("time_to_rug_hours", 72),
                "max_drawdown_pct": token.get("max_drawdown_pct", 20),
                "pump_2x_label": token.get("pump_2x_label", 0),
            })
    return pd.DataFrame(rows)

# Main collection flow
print("="*60)
print("Starting mint collection (FIX 1: DexScreener + Helius, no text search)")
print("="*60)

mint_list = collect_mints_from_apis(target_tokens=200)

if not mint_list:
    print("No tokens found via APIs. Creating synthetic samples for pipeline verification...")
    mints_set = {f"synthetic_mint_{i}pump" for i in range(200)}
    mint_list = [{"mint": m, "deployer": f"deployer_{i % 10}", "liquidity_usd": 0, "fdv": 0} for i, m in enumerate(mints_set)]

print(f"\nEnriching {len(mint_list)} tokens with Helius transactions...")
mint_list = enrich_with_helius_transactions(mint_list)

print(f"\nEnriching {len(mint_list)} tokens with Birdeye price data...")
mint_list = enrich_with_birdeye_price(mint_list)

print(f"\nBuilding raw_df from {len(mint_list)} tokens...")
raw_df = build_raw_df_from_tokens(mint_list)

if not raw_df.empty:
    raw_df.to_parquet(RAW_DATA_PATH, index=False)

rug_count = raw_df["rug_label"].sum() if "rug_label" in raw_df.columns else 0
total_tokens = raw_df["mint"].nunique()
print(f"Dataset ready: {len(raw_df)} rows, {total_tokens} tokens, {rug_count} rugs ({rug_count/max(total_tokens,1)*100:.1f}%)")

Starting mint collection (FIX 1: DexScreener + Helius, no text search)
Fetching DexScreener Raydium pairs...
  Got 0 pairs
Fetching DexScreener token profiles...
  Got 30 profiles
Fetching Helius pump.fun migration transactions...
  Got 76 migrations
Total unique mints collected: 94

Enriching 94 tokens with Helius transactions...


Helius txs:   0%|          | 0/94 [00:00<?, ?it/s]

Rate limited — waiting 1.1s
Rate limited — waiting 2.6s
Rate limited — waiting 4.2s
Rate limited — waiting 9.0s


KeyboardInterrupt: 

In [5]:
# Cell 3b: Enrich raw_df with ground-truth rug events from confirmed_rugs.csv (optional)
from pathlib import Path as _Path

CONFIRMED_RUGS_PATHS = [
    CONFIG['DRIVE_BASE'] + '/confirmed_rugs.csv',
    '/content/confirmed_rugs.csv',
    'ml/data/confirmed_rugs.csv',
]

confirmed_path = next(
    (p for p in CONFIRMED_RUGS_PATHS
     if os.path.exists(p)),
    None
)

if confirmed_path is None:
    print("confirmed_rugs.csv not found — skipping")
    print("Labels will come from Helius price data")
else:
    confirmed = pd.read_csv(confirmed_path, parse_dates=["rug_date"], keep_default_na=False)
    expected_cols = {"mint", "rug_date", "amount_stolen_usd", "rug_method", "pump_before_rug", "liquidity_drain_pct"}
    missing = expected_cols - set(confirmed.columns)
    if missing:
        print(f"confirmed_rugs.csv missing columns: {sorted(missing)}; skipping enrichment")
    else:
        # Normalize types
        confirmed["mint"] = confirmed["mint"].astype(str)
        confirmed["pump_before_rug"] = confirmed["pump_before_rug"].astype(bool)
        # Ensure raw_df timestamps are datetimes
        raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], utc=True)

        # Keep only confirmed mints present in raw data
        present = confirmed[confirmed["mint"].isin(raw_df["mint"])].copy()
        LABEL_OVERRIDES = {}
        enriched_mints = []
        deployer_counts = {}

        for _, crow in present.iterrows():
            mint = str(crow["mint"])
            token_rows = raw_df[raw_df["mint"] == mint].sort_values("timestamp")
            if token_rows.empty:
                continue
            first_swap = pd.to_datetime(token_rows["timestamp"].min(), utc=True)
            rug_date = pd.to_datetime(crow["rug_date"], utc=True)
            time_to_rug_hours = max(0.0, (rug_date - first_swap).total_seconds() / 3600.0)
            pump_label = 1.0 if bool(crow.get("pump_before_rug", False)) else 0.0
            raw_liq = safe_float(crow.get("liquidity_drain_pct", 0.0), 0.0)
            if 0.0 <= raw_liq <= 1.0:
                liquidity_pct = float(raw_liq * 100.0)
            else:
                liquidity_pct = float(raw_liq)

            LABEL_OVERRIDES[mint] = {
                "rug_label": 1.0,
                "time_to_rug_hours": float(time_to_rug_hours),
                "override_max_drawdown_pct": float(liquidity_pct),
                "pump_2x_label": float(pump_label),
            }
            enriched_mints.append(mint)
            deployer = str(token_rows.iloc[0].get("deployer", ""))
            deployer_counts[deployer] = deployer_counts.get(deployer, 0) + 1

        known_deployers = {d for d, c in deployer_counts.items() if c >= 2}
        raw_df["known_rugger_deployer"] = raw_df["deployer"].astype(str).isin(known_deployers)

        raw_df["override_rug_label"] = raw_df["mint"].map(lambda m: LABEL_OVERRIDES.get(str(m), {}).get("rug_label", np.nan))
        raw_df["override_time_to_rug_hours"] = raw_df["mint"].map(lambda m: LABEL_OVERRIDES.get(str(m), {}).get("time_to_rug_hours", np.nan))
        raw_df["override_max_drawdown_pct"] = raw_df["mint"].map(lambda m: LABEL_OVERRIDES.get(str(m), {}).get("override_max_drawdown_pct", np.nan))
        raw_df["override_pump_2x_label"] = raw_df["mint"].map(lambda m: LABEL_OVERRIDES.get(str(m), {}).get("pump_2x_label", np.nan))

        total_mints = int(raw_df["mint"].nunique())
        enriched_count = len(set(enriched_mints))
        label_override_rate = enriched_count / max(total_mints, 1)

        def _quick_infer_rug(token_df):
            decision_ts = pd.to_datetime(token_df.iloc[0]["graduation_timestamp"], utc=True) + pd.Timedelta(seconds=CONFIG["ENTRY_DELAY_SECONDS"])
            horizon_end = decision_ts + pd.Timedelta(hours=CONFIG["LABEL_WINDOW_HOURS"])
            future = token_df[(token_df["timestamp"] >= decision_ts) & (token_df["timestamp"] <= horizon_end)].copy()
            if future.empty:
                return 0.0
            future["price_sol"] = future["price_sol"].ffill().bfill()
            future = future[np.isfinite(future["price_sol"])].copy()
            if future.empty:
                return 0.0
            entry_price = token_df[token_df["timestamp"] <= decision_ts]["price_sol"].ffill().bfill()
            if entry_price.empty or not np.isfinite(entry_price.iloc[-1]):
                return 0.0
            entry_price = float(entry_price.iloc[-1])
            future["drawdown_pct"] = (entry_price - future["price_sol"]) / entry_price
            max_drawdown_pct = clamp(float(future["drawdown_pct"].max()), 0.0, 1.0) * 100.0
            price_drop_mask = future["drawdown_pct"] >= CONFIG["PRICE_DROP_RUG_THRESHOLD"]
            liq_proxy = future["sol_amount"].rolling(12, min_periods=1).sum()
            liquidity_drop_mask = (float(token_df.iloc[0].get("initial_liquidity_sol", 0.0)) - liq_proxy) / max(float(token_df.iloc[0].get("initial_liquidity_sol", 0.0)), 1e-9) >= CONFIG["LIQUIDITY_DROP_RUG_THRESHOLD"]
            rug_rows = future[price_drop_mask & liquidity_drop_mask]
            return 1.0 if not rug_rows.empty else 0.0

        orig_labels = []
        for mint in set(enriched_mints):
            tdf = raw_df[raw_df["mint"] == mint].sort_values("timestamp")
            orig_labels.append(_quick_infer_rug(tdf))
        class_before = float(sum(orig_labels)) / max(len(orig_labels), 1) if orig_labels else 0.0
        class_after = 1.0

        print({
            "enriched_mints": enriched_count,
            "total_mints": total_mints,
            "label_override_rate": label_override_rate,
            "class_balance_before_among_enriched": class_before,
            "class_balance_after_among_enriched": class_after,
        })

        raw_df.to_parquet(RAW_DATA_PATH, index=False)

confirmed_rugs.csv not found — skipping
Labels will come from Helius price data


In [6]:
# Cell 3c: Dataset augmentation for small datasets
def augment_dataset(df, target=1500):
    """If collected tokens < 1000, generate synthetic variations to reach target."""
    if len(df) >= target:
        return df

    needed = target - len(df)

    # Only add noise to continuous features
    continuous = [
        'lpBurnGap', 'transferTaxPct',
        'topHolderPct', 'devHoldPct',
        'volatility1m', 'lowLiquidity',
        'lowBuyers', 'rugcheckLpUnlocked',
        'rugcheckDangerSignals'
    ]
    # Never modify binary features or labels
    binary = [
        'mutableMetadata', 'mintAuthority',
        'freezeAuthority', 'rug_label',
        'pump_2x_label'
    ]

    rows = []
    for i in range(needed):
        base = df.sample(1).iloc[0].copy()
        for col in continuous:
            if col in base.index:
                noise = np.random.normal(0, 0.02)
                base[col] = float(
                    np.clip(base[col] + noise, 0, 1)
                )
        base['is_augmented'] = True
        rows.append(base)

    augmented = pd.concat(
        [df, pd.DataFrame(rows)],
        ignore_index=True
    )
    augmented.to_parquet(RAW_DATA_PATH, index=False)

    print(f"Augmented: {len(df)} → {len(augmented)}")
    print(f"Rug rate: {augmented['rug_label'].mean():.1%}")
    return augmented

raw_df = augment_dataset(raw_df)
print(f"Final dataset: {len(raw_df)} rows, {raw_df['mint'].nunique()} tokens")

NameError: name 'raw_df' is not defined

In [ ]:
FEATURES_PATH = Path(CONFIG["FEATURES_PATH"])

def compute_price_sol(row: pd.Series) -> float:
    token_amount = safe_float(row.get("token_amount"), 0.0)
    sol_amount = safe_float(row.get("sol_amount"), 0.0)
    return sol_amount / token_amount if token_amount > 0 and sol_amount > 0 else np.nan

def build_sequence(token_df: pd.DataFrame, decision_ts: pd.Timestamp) -> list[list[float]]:
    bucket_minutes = CONFIG["TIME_BUCKET_MINUTES"]
    history = token_df[token_df["timestamp"] <= decision_ts].copy()
    if history.empty:
        return [[0.0] * 6 for _ in range(CONFIG["SEQUENCE_LENGTH"])]

    history["bucket"] = history["timestamp"].dt.floor(f"{bucket_minutes}min")
    grouped = []
    prev_price = None
    for bucket, chunk in history.groupby("bucket", sort=True):
        holders = chunk["buyer"].nunique() + chunk["seller"].nunique()
        liquidity = safe_float(chunk["initial_liquidity_sol"].iloc[0], 0.0)
        volume = chunk["sol_amount"].sum()
        buy_count = chunk["buyer"].notna().sum()
        sell_count = chunk["seller"].notna().sum()
        ratio = buy_count / max(sell_count, 1)
        prices = chunk["price_sol"].dropna()
        current_price = float(prices.iloc[-1]) if not prices.empty else (prev_price or 0.0)
        velocity = 0.0 if prev_price in (None, 0.0) else (current_price - prev_price) / max(prev_price, 1e-9)
        prev_price = current_price
        grouped.append([float(holders), float(liquidity), float(volume), float(ratio), float(velocity), float(len(chunk))])

    grouped = grouped[-CONFIG["SEQUENCE_LENGTH"]:]
    while len(grouped) < CONFIG["SEQUENCE_LENGTH"]:
        grouped.insert(0, [0.0, 0.0, 0.0, 1.0, 0.0, 0.0])
    return grouped

def compute_labels(token_df: pd.DataFrame, decision_ts: pd.Timestamp, entry_price: float, initial_liquidity_sol: float) -> dict[str, Any]:
    # After removing rugcheck, if there's no Birdeye data, we cannot infer rugPullRisk
    # Default to a non-rug label if no other information is available.
    return {
        "rug_label": 0.0,
        "time_to_rug_hours": 72.0,
        "max_drawdown_pct": 20.0,
        "pump_2x_label": 0.0,
        "inferred_label": True
    }

def build_feature_rows(raw_df: pd.DataFrame, force_refresh: bool = False) -> pd.DataFrame:
    if FEATURES_PATH.exists() and not force_refresh:
        return pd.read_parquet(FEATURES_PATH)

    df = raw_df.copy()
    df["price_sol"] = df.apply(compute_price_sol, axis=1)

    feature_rows = []
    for mint, token_df in df.groupby("mint"):
        token_df = token_df.sort_values("timestamp").reset_index(drop=True)
        meta = token_df.iloc[0]
        decision_ts = pd.Timestamp(meta["graduation_timestamp"]) + pd.Timedelta(seconds=CONFIG["ENTRY_DELAY_SECONDS"])

        sequence = build_sequence(token_df, decision_ts)
        # Fix nested sequence validation
        has_sequence = any(any(abs(v) > 1e-9 for v in step) for step in sequence)

        labels = compute_labels(token_df, decision_ts, 1.0, 20.0)

        # Dynamically build tabular features based on CONFIG["TABULAR_FEATURES"]
        tabular_values = [safe_float(meta.get(feature, 0.0)) for feature in CONFIG["TABULAR_FEATURES"]]

        feature_rows.append({
            "mint": mint,
            "deployer": str(meta["deployer"]),
            "decision_timestamp": decision_ts,
            "tabular": tabular_values,
            "sequence": sequence,
            "rug_label": labels["rug_label"],
            "time_to_rug_hours": labels["time_to_rug_hours"],
            "max_drawdown_pct": labels["max_drawdown_pct"],
            "pump_2x_label": labels["pump_2x_label"]
        })

    f_df = pd.DataFrame(feature_rows)
    f_df.to_parquet(FEATURES_PATH, index=False)
    return f_df

features_df = build_feature_rows(raw_df, force_refresh=True)
features_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

def make_split(df):
    """Temporal split with fallback to random stratified if <1000 tokens or single-class splits."""
    total = len(df)
    rug_rate = df['rug_label'].mean()

    # Date boundaries for temporal split
    TRAIN_END_TS = pd.Timestamp(CONFIG["TRAIN_END"])
    VAL_END_TS = pd.Timestamp(CONFIG["VAL_END"])

    # Check if temporal split will work
    train_mask = df['graduation_timestamp'] <= TRAIN_END_TS
    val_mask = (df['graduation_timestamp'] > TRAIN_END_TS) & \
               (df['graduation_timestamp'] <= VAL_END_TS)
    test_mask = df['graduation_timestamp'] > VAL_END_TS

    train_df = df[train_mask]
    val_df = df[val_mask]
    test_df = df[test_mask]

    # Check each split has both classes
    splits_ok = all(
        df_split['rug_label'].nunique() >= 2
        for df_split in [train_df, val_df, test_df]
        if len(df_split) > 0
    )

    if not splits_ok or total < 1000:
        print("WARNING: Using random stratified split")
        print(f"  Reason: {'small dataset' if total < 1000 else 'single class in split'}")
        print(f"  Total tokens: {total}")
        print("  NOTE: Retrain with temporal split once")
        print("  you have 1000+ tokens for production")

        train_df, temp = train_test_split(
            df, test_size=0.2,
            stratify=df['rug_label'],
            random_state=42
        )
        val_df, test_df = train_test_split(
            temp, test_size=0.5,
            stratify=temp['rug_label'],
            random_state=42
        )

    for name, split in [('train', train_df),
                         ('val', val_df),
                         ('test', test_df)]:
        print(f"{name}: {len(split)} samples, "
              f"rug={split['rug_label'].mean():.1%}")

    return train_df, val_df, test_df

def build_deployer_vocab(df):
    deployers = sorted(df["deployer"].astype(str).unique())
    vocab = {"__UNK__": CONFIG["UNKNOWN_DEPLOYER_ID"]}
    vocab.update({d: i + 1 for i, d in enumerate(deployers)})
    return vocab

def attach_deployer_ids(df, vocab):
    df = df.copy()
    df["deployer_id"] = df["deployer"].astype(str).map(lambda v: vocab.get(v, CONFIG["UNKNOWN_DEPLOYER_ID"]))
    return df

# Build features_df first (from raw_df) if not already done
# Re-use build_feature_rows from Cell 4
# Note: features_df should already exist from Cell 4 execution
if "features_df" not in dir():
    print("features_df not found, building from raw_df...")
    features_df = build_feature_rows(raw_df, force_refresh=True)

train_df, val_df, test_df = make_split(features_df)
deployer_vocab = build_deployer_vocab(train_df)
train_df = attach_deployer_ids(train_df, deployer_vocab)
val_df = attach_deployer_ids(val_df, deployer_vocab)
test_df = attach_deployer_ids(test_df, deployer_vocab)

print(f"\nSplits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
print(f"Rug count in train: {train_df['rug_label'].sum()}")

In [ ]:
class RugDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.frame.iloc[idx]
        # Robust sequence: handles parquet-serialized nested lists
        seq = row["sequence"]
        if isinstance(seq, np.ndarray):
            seq = seq.tolist()
        seq_tensor = torch.tensor(seq, dtype=torch.float32)
        if seq_tensor.ndim == 1:
            seq_tensor = seq_tensor.view(CONFIG["SEQUENCE_LENGTH"], -1)
        return {
            "tabular": torch.tensor(row["tabular"], dtype=torch.float32),
            "sequence": seq_tensor,
            "deployer_id": torch.tensor(row["deployer_id"], dtype=torch.long),
            "rug_label": torch.tensor(row["rug_label"], dtype=torch.float32),
            "time_to_rug_hours": torch.tensor(min(float(row["time_to_rug_hours"]), 72.0), dtype=torch.float32),
            "max_drawdown_pct": torch.tensor(float(row["max_drawdown_pct"]), dtype=torch.float32),
            "pump_2x_label": torch.tensor(float(row["pump_2x_label"]), dtype=torch.float32),
        }


def collate_batch(items: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:
    return {key: torch.stack([item[key] for item in items]) for key in items[0]}


def build_sampler(frame: pd.DataFrame):
    counts = frame["rug_label"].value_counts().to_dict()
    pos = counts.get(1.0, 0)
    neg = counts.get(0.0, 0)
    ratio = neg / max(pos, 1)
    if ratio <= 5:
        return None
    pos_weight = ratio
    weights = np.where(frame["rug_label"].to_numpy() >= 0.5, pos_weight, 1.0)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)

train_dataset = RugDataset(train_df)
val_dataset = RugDataset(val_df)
test_dataset = RugDataset(test_df)
train_sampler = build_sampler(train_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    sampler=train_sampler,
    shuffle=train_sampler is None,
    collate_fn=collate_batch,
)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_batch)

print({
    "train_batches": len(train_loader),
    "val_batches": len(val_loader),
    "test_batches": len(test_loader),
    "weighted_sampler": train_sampler is not None,
})

In [3]:
# Input contract for parity with TS inference/export:
# - tabular: [batch, N] where N is number of tabular features
# - sequence: [batch, 16, 6]
# - deployer_id: [batch]
# Outputs:
# - rug_prob: [batch, 1]
# - time_to_rug_hours: [batch, 1]
# - max_drawdown_pct: [batch, 1]
# - pump_2x_prob: [batch, 1]

class MCDropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p
        self.force_dropout = False

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return F.dropout(value, p=self.p, training=self.training or self.force_dropout)


class SequenceEncoder(nn.Module):
    def __init__(self, input_dim: int = 6, hidden_dim: int = 16, num_layers: int = 2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        if sequence.ndim != 3:
            raise ValueError("sequence tensor must be [batch, 16, 6]")
        _, hidden = self.gru(sequence)
        return hidden[-1]


class RugRiskNet(nn.Module):
    def __init__(
        self,
        tabular_dim: int = len(CONFIG["TABULAR_FEATURES"]),
        num_deployers: int = 1,
        deployer_dim: int = 32,
        sequence_dim: int = 16,
        feature_mean: np.ndarray | None = None,
        feature_std: np.ndarray | None = None,
    ):
        super().__init__()
        input_dim = tabular_dim + deployer_dim + sequence_dim
        self.register_buffer("feature_mean", torch.tensor(feature_mean if feature_mean is not None else np.zeros(tabular_dim), dtype=torch.float32))
        self.register_buffer("feature_std", torch.tensor(feature_std if feature_std is not None else np.ones(tabular_dim), dtype=torch.float32))
        self.deployer_embedding = nn.Embedding(num_deployers, deployer_dim)
        nn.init.normal_(self.deployer_embedding.weight, mean=0.0, std=0.02)
        self.sequence_encoder = SequenceEncoder(input_dim=6, hidden_dim=sequence_dim, num_layers=2)

        self.input_skip = nn.Linear(input_dim, 128)
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            MCDropout(CONFIG["DROPOUT_1"]),
        )
        self.skip2 = nn.Linear(128, 64)
        self.block2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            MCDropout(CONFIG["DROPOUT_2"]),
        )
        self.block3 = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.rug_head = nn.Linear(32, 1)
        self.time_head = nn.Linear(32, 1)
        self.drawdown_head = nn.Linear(32, 1)
        self.pump_head = nn.Linear(32, 1)

    def encode_backbone(self, tabular: torch.Tensor, deployer_id: torch.Tensor, sequence: torch.Tensor) -> torch.Tensor:
        tabular = (tabular - self.feature_mean) / torch.clamp(self.feature_std, min=1e-6)
        deployer = self.deployer_embedding(deployer_id.long())
        temporal = self.sequence_encoder(sequence)
        x = torch.cat([tabular, deployer, temporal], dim=1)
        h1 = self.block1(x) + self.input_skip(x)
        h2 = self.block2(h1) + self.skip2(h1)
        return self.block3(h2)

    def forward_logits(self, tabular: torch.Tensor, sequence: torch.Tensor, deployer_id: torch.Tensor):
        z = self.encode_backbone(tabular, deployer_id, sequence)
        rug_logits = self.rug_head(z)
        time_to_rug = torch.clamp(F.relu(self.time_head(z)), max=72.0)
        max_drawdown = torch.clamp(100.0 * torch.sigmoid(self.drawdown_head(z)), min=0.0, max=100.0)
        pump_logits = self.pump_head(z)
        return rug_logits, time_to_rug, max_drawdown, pump_logits

    def forward(self, tabular: torch.Tensor, sequence: torch.Tensor, deployer_id: torch.Tensor):
        rug_logits, time_to_rug, max_drawdown, pump_logits = self.forward_logits(tabular, sequence, deployer_id)
        return torch.sigmoid(rug_logits), time_to_rug, max_drawdown, torch.sigmoid(pump_logits)

feature_mean = np.asarray(train_df["tabular"].tolist(), dtype=np.float32).mean(axis=0)
feature_std = np.clip(np.asarray(train_df["tabular"].tolist(), dtype=np.float32).std(axis=0), 1e-4, None)
model = RugRiskNet(num_deployers=len(deployer_vocab), feature_mean=feature_mean, feature_std=feature_std).to(DEVICE)
print(model)
# XGBoost classifier for tabular features
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=CONFIG['XGB_N_ESTIMATORS'],
    max_depth=4,
    learning_rate=CONFIG['XGB_LEARNING_RATE'],
    subsample=CONFIG['XGB_SUBSAMPLE'],
    colsample_bytree=0.4,
    reg_alpha=1.0,
    reg_lambda=2.0,
    use_label_encoder=False,
    eval_metric='auc',
    tree_method='hist',
    device='cuda'
)

TABULAR_FEATURE_NAMES = CONFIG["TABULAR_FEATURES"]

NameError: name 'train_df' is not defined

In [4]:
# Prepare tabular data for XGBoost using all current TABULAR_FEATURES
XGB_FEATURE_NAMES = TABULAR_FEATURE_NAMES
X_train_tabular = np.asarray(train_df["tabular"].tolist(), dtype=np.float32)
X_train_xgb = X_train_tabular  # Use all tabular features
y_train_rug = train_df["rug_label"].to_numpy(dtype=np.float32)
X_val_tabular = np.asarray(val_df["tabular"].tolist(), dtype=np.float32)
X_val_xgb = X_val_tabular # Use all tabular features
y_val_rug = val_df["rug_label"].to_numpy(dtype=np.float32)

print("Training XGBoost (using all current tabular features)...")
xgb_clf.fit(
    X_train_xgb, y_train_rug,
    eval_set=[(X_val_xgb, y_val_rug)],
    verbose=False
)

importances = dict(zip(XGB_FEATURE_NAMES, xgb_clf.feature_importances_))
print("\nTop 5 features by importance:")
for feat, imp in sorted(importances.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {feat}: {imp:.4f}")

print("\nTraining PyTorch RugRiskNet...")
optimizer = torch.optim.AdamW(
    [{"params": [p for n, p in model.named_parameters() if "embedding" not in n], "lr": CONFIG["LEARNING_RATE"]},
     {"params": model.deployer_embedding.parameters(), "lr": CONFIG["LEARNING_RATE"] * 0.1}],
    weight_decay=CONFIG["WEIGHT_DECAY"]
)

best_val_auc = -1.0
for epoch in range(1, 21):
    model.train()
    train_losses = []
    for batch in train_loader:
        batch = move_batch(batch)
        optimizer.zero_grad()
        loss, _ = multitask_loss(model, batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics = evaluate_epoch(model, val_loader)
    if val_metrics["rug_auc"] > best_val_auc:
        best_val_auc = val_metrics["rug_auc"]
        torch.save({"state_dict": model.state_dict(), "feature_mean": feature_mean, "feature_std": feature_std, "deployer_vocab": deployer_vocab, "history": [], "best_val_auc": best_val_auc}, CONFIG["BEST_CHECKPOINT_PATH"])

    if epoch % 5 == 0:
        print(f"Epoch {epoch}: Train Loss={np.mean(train_losses):.4f}, Val AUC={val_metrics['rug_auc']:.4f}")

print(f"\nTraining Complete. Best Val AUC: {best_val_auc:.4f}")

NameError: name 'TABULAR_FEATURE_NAMES' is not defined

In [ ]:
checkpoint = torch.load(CONFIG["BEST_CHECKPOINT_PATH"], map_location=DEVICE, weights_only=False)
model = RugRiskNet(
    num_deployers=len(checkpoint["deployer_vocab"]),
    feature_mean=np.asarray(checkpoint["feature_mean"], dtype=np.float32),
    feature_std=np.asarray(checkpoint["feature_std"], dtype=np.float32),
).to(DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# Ensure we are evaluating on the full test set
@torch.no_grad()
def collect_predictions(model: RugRiskNet, loader: DataLoader):
    out = {"rug": [], "rug_y": [], "pump_y": [], "time": [], "time_y": [], "drawdown": [], "drawdown_y": []}
    for batch in loader:
        batch = move_batch(batch)
        rug_prob, time_pred, drawdown_pred, pump_prob = model(batch["tabular"], batch["sequence"], batch["deployer_id"])
        out["rug"].extend(rug_prob.view(-1).cpu().numpy().tolist())
        out["rug_y"].extend(batch["rug_label"].view(-1).cpu().numpy().tolist())
        out["pump_y"].extend(batch["pump_2x_label"].view(-1).cpu().numpy().tolist())
        out["time"].extend(time_pred.view(-1).cpu().numpy().tolist())
        out["time_y"].extend(batch["time_to_rug_hours"].view(-1).cpu().numpy().tolist())
        out["drawdown"].extend(drawdown_pred.view(-1).cpu().numpy().tolist())
        out["drawdown_y"].extend(batch["max_drawdown_pct"].view(-1).cpu().numpy().tolist())
    return {k: np.array(v) for k, v in out.items()}

preds = collect_predictions(model, test_loader)
torch_probs = preds["rug"]
y_test_rug = preds["rug_y"]

# Prepare tabular data strictly from the same test set used in the loader
X_test_tabular = np.asarray(test_df["tabular"].tolist(), dtype=np.float32)
X_test_xgb = X_test_tabular[:, 1:]  # drop rugPullRisk
xgb_probs = xgb_clf.predict_proba(X_test_xgb)[:, 1]

# Parity Check on shapes
if len(xgb_probs) != len(torch_probs):
    print(f"Warning: Shape mismatch fixed. Truncating/Aligning: XGB({len(xgb_probs)}) vs Torch({len(torch_probs)})")
    min_len = min(len(xgb_probs), len(torch_probs))
    xgb_probs = xgb_probs[:min_len]
    torch_probs = torch_probs[:min_len]
    y_test_rug = y_test_rug[:min_len]

ensemble_probs = (
    xgb_probs * CONFIG["XGB_ENSEMBLE_WEIGHT"] +
    torch_probs * CONFIG["TORCH_ENSEMBLE_WEIGHT"]
)

ensemble_auc = roc_auc_score(y_test_rug, ensemble_probs) if len(np.unique(y_test_rug)) > 1 else 0.5

print("=" * 40)
print(f"Ensemble AUC: {ensemble_auc:.4f} (Gate: {CONFIG['AUC_QUALITY_GATE']})")
print("=" * 40)

QUALITY_GATE_PASSED = ensemble_auc >= CONFIG["AUC_QUALITY_GATE"]

# Store metadata for export
model_meta = {
    "version": 1,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "test": {
            "ensemble_auc": float(ensemble_auc),
            "torch_auc": float(roc_auc_score(y_test_rug, torch_probs) if len(np.unique(y_test_rug)) > 1 else 0.5),
            "xgb_auc": float(roc_auc_score(y_test_rug, xgb_probs) if len(np.unique(y_test_rug)) > 1 else 0.5)
        }
    }
}

if QUALITY_GATE_PASSED:
    print("Quality gate passed; Export is enabled.")
else:
    print(f"MODEL FAILED QUALITY GATE ({ensemble_auc:.4f})")

In [5]:
if not globals().get("QUALITY_GATE_PASSED", False):
    raise RuntimeError("MODEL FAILED QUALITY GATE — DO NOT EXPORT")

import onnxruntime as ort
import json
import torch
import os
import xgboost as xgb
from torch import nn
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

# Ensure directory exists
os.makedirs(os.path.dirname(CONFIG["ONNX_OUTPUT_PATH"]), exist_ok=True)

def set_force_dropout(module: nn.Module, enabled: bool) -> None:
    for child in module.modules():
        if isinstance(child, MCDropout):
            child.force_dropout = enabled

# Save vocab and metadata
id_to_deployer = {int(idx): deployer for deployer, idx in checkpoint["deployer_vocab"].items()}
with open(CONFIG["DEPLOYER_LOOKUP_PATH"], "w", encoding="utf-8") as handle:
    json.dump(id_to_deployer, handle, indent=2)

with open(CONFIG["MODEL_META_PATH"], "w", encoding="utf-8") as handle:
    json.dump(model_meta, handle, indent=2)

# 1. Export PyTorch Model - Using legacy exporter for GRU stability
model.eval()
set_force_dropout(model, True)
example = next(iter(test_loader))
example_tabular = example["tabular"][:1].to(DEVICE)
example_sequence = example["sequence"][:1].to(DEVICE)
example_deployer = example["deployer_id"][:1].to(DEVICE)

try:
    torch.onnx.export(
        model,
        (example_tabular, example_sequence, example_deployer),
        CONFIG["ONNX_OUTPUT_PATH"],
        input_names=["tabular", "sequence", "deployer_id"],
        output_names=["rug_prob", "time_to_rug", "max_drawdown", "pump_2x"],
        dynamic_axes={
            "tabular": {0: "batch"},
            "sequence": {0: "batch"},
            "deployer_id": {0: "batch"},
            "rug_prob": {0: "batch"},
            "time_to_rug": {0: "batch"},
            "max_drawdown": {0: "batch"},
            "pump_2x": {0: "batch"},
        },
        opset_version=15,
        do_constant_folding=True,
        dynamo=False
    )
finally:
    set_force_dropout(model, False)

# 2. Export XGBoost Model
initial_types = [("tabular_input", FloatTensorType([None, len(XGB_FEATURE_NAMES)]))]
xgb_onnx = onnxmltools.convert_xgboost(xgb_clf, initial_types=initial_types, target_opset=15)
with open(CONFIG["XGB_MODEL_OUT"], "wb") as f:
    f.write(xgb_onnx.SerializeToString())
    f.flush()
    os.fsync(f.fileno())

print(f"Successfully exported:\n - {CONFIG['ONNX_OUTPUT_PATH']}\n - {CONFIG['XGB_MODEL_OUT']}")
print(f"File check: {os.path.exists(CONFIG['XGB_MODEL_OUT'])}")

RuntimeError: MODEL FAILED QUALITY GATE — DO NOT EXPORT

In [ ]:
import onnxruntime as ort
import numpy as np
from sklearn.metrics import roc_auc_score
import os

if not os.path.exists(CONFIG['ONNX_OUTPUT_PATH']) or not os.path.exists(CONFIG['XGB_MODEL_OUT']):
    print("Error: ONNX files missing. Run export cell first.")
else:
    torch_sess = ort.InferenceSession(CONFIG['ONNX_OUTPUT_PATH'], providers=['CPUExecutionProvider'])
    xgb_sess = ort.InferenceSession(CONFIG['XGB_MODEL_OUT'], providers=['CPUExecutionProvider'])

    torch_onnx_probs = []
    xgb_onnx_probs = []
    true_labels = []

    for batch in test_loader:
        inputs = {
            "tabular": batch["tabular"].numpy().astype(np.float32),
            "sequence": batch["sequence"].numpy().astype(np.float32),
            "deployer_id": batch["deployer_id"].numpy().astype(np.int64),
        }
        t_out = torch_sess.run(None, inputs)
        torch_onnx_probs.extend(t_out[0].flatten().tolist())

        x_inputs = {"tabular_input": batch["tabular"].numpy().astype(np.float32)}
        x_out = xgb_sess.run(None, x_inputs)
        # Note: skl2onnx XGBClassifier output [0] is labels, [1] is probabilities
        xgb_onnx_probs.extend(x_out[1][:, 1].flatten().tolist())
        true_labels.extend(batch["rug_label"].numpy().flatten().tolist())

    ensemble_onnx_probs = (
        np.array(xgb_onnx_probs) * CONFIG["XGB_ENSEMBLE_WEIGHT"] +
        np.array(torch_onnx_probs) * CONFIG["TORCH_ENSEMBLE_WEIGHT"]
    )

    onnx_auc = roc_auc_score(true_labels, ensemble_onnx_probs) if len(set(true_labels)) > 1 else 0.5
    print(f"ONNX Ensemble AUC: {onnx_auc:.4f}")

    if 'ensemble_auc' in globals():
        diff = abs(onnx_auc - ensemble_auc)
        print(f"Parity Delta: {diff:.8f}")
        if diff < 1e-4:
            print("✅ ONNX Parity Verified.")
        else:
            print("❌ Parity Delta exceeds tolerance.")

In [6]:
import onnx
import matplotlib.pyplot as plt
import numpy as np
import os

if os.path.exists(CONFIG['XGB_MODEL_OUT']):
    # Extract feature importance from the original model for verification
    importances = xgb_clf.feature_importances_
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10, 6))
    plt.title("XGBoost Feature Importance (using all current tabular features)")
    plt.bar(range(len(importances)), importances[indices], align="center")
    plt.xticks(range(len(importances)), [XGB_FEATURE_NAMES[i] for i in indices], rotation=90)
    plt.ylabel("Gini Importance")
    plt.tight_layout()
    plt.show()

    print("Feature Rankings (XGBoost, using all current tabular features):")
    for i in indices:
        print(f"{XGB_FEATURE_NAMES[i]}: {importances[i]:.4f}")
else:
    print(f"Error: {CONFIG['XGB_MODEL_OUT']} still not found. Check Drive permissions/path.")

ModuleNotFoundError: No module named 'onnx'

## Deployment Checklist

[ ] Test AUC >= 0.65 (quality gate passed)
[ ] `rug_model.onnx` downloaded from Kaggle output
[ ] `deployer_lookup.json` downloaded and placed in `models/`
[ ] `models/rug_model_meta.json` updated with new `trained_at`, `train_period`, `val_period`, `test_period`, `test_auc`
[ ] `ALLOW_ONNX_RUG_MODEL=true` set in `.env` (was disabled pending retraining — this is now re-enabled)
[ ] `RUG_MODEL_PATH=models/rug_model.onnx` set in `.env`
[ ] Bot restarted and `rug_prob` scores logged for first 100 tokens to verify live inference is sane

In [ ]:
print("Checking Helius API quota usage...")
try:
    # Use the helius_get function to fetch usage data
    usage_data = helius_get(path="/v0/users/me/usage", params={})
    print(json.dumps(usage_data, indent=2))
    # You might want to parse and display specific fields more clearly
    if usage_data and 'usage' in usage_data and 'dailyLimits' in usage_data['usage']:
        daily_limits = usage_data['usage']['dailyLimits']
        current_usage = usage_data['usage']['currentUsage']
        print(f"\n--- Helius Daily Usage Summary ---")
        for endpoint, limits in daily_limits.items():
            used = current_usage.get(endpoint, 0)
            limit = limits.get('limit', 'N/A')
            remaining = 'N/A' if limit == 'N/A' else limit - used
            print(f"  {endpoint}: Used {used}, Limit {limit}, Remaining {remaining}")
    else:
        print("Could not parse Helius usage data. Response format might have changed or data is incomplete.")
except Exception as e:
    print(f"Error checking Helius API quota: {e}")